In [ ]:
!pip install -q peft transformers datasets

In [ ]:
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from datasets import load_dataset

In [ ]:
model = TFAutoModel.from_pretrained("bert-base-uncased")

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
inputs = tokenizer(['Hello world', 'Hi how are you'], padding=True, truncation=True,
                  return_tensors='tf')
inputs


{'input_ids': <tf.Tensor: shape=(2, 6), dtype=int32, numpy=
array([[ 101, 7592, 2088,  102,    0,    0],
       [ 101, 7632, 2129, 2024, 2017,  102]], dtype=int32)>, 'token_type_ids': <tf.Tensor: shape=(2, 6), dtype=int32, numpy=
array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(2, 6), dtype=int32, numpy=
array([[1, 1, 1, 1, 0, 0],
       [1, 1, 1, 1, 1, 1]], dtype=int32)>}

In [ ]:
output = model(inputs)
output

TFBaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=<tf.Tensor: shape=(2, 6, 768), dtype=float32, numpy=
array([[[-0.1688833 ,  0.13606358, -0.13940008, ..., -0.6251125 ,
          0.05217254,  0.36714533],
        [-0.36327448,  0.14121935,  0.8799882 , ...,  0.10433007,
          0.28875723,  0.3726793 ],
        [-0.6985944 , -0.69879735,  0.06450275, ..., -0.22103691,
          0.00986822, -0.5939794 ],
        [ 0.8309832 ,  0.1236674 , -0.1511907 , ...,  0.10309584,
         -0.67792666, -0.26285216],
        [-0.40266573, -0.01928258,  0.57325053, ..., -0.2065685 ,
          0.0233856 ,  0.2012629 ],
        [-0.6228409 , -0.27453488,  0.18117625, ..., -0.12944855,
         -0.03839095, -0.057332  ]],

       [[ 0.0928655 , -0.02636387, -0.12239331, ..., -0.21063554,
          0.17386377,  0.17250937],
        [ 0.40742067, -0.05930961,  0.5523465 , ..., -0.6790571 ,
          0.65557456, -0.29456607],
        [-0.21155313, -0.68586326, -0.46280748, ...,  0.15278433

In [ ]:
dataset_name = "twitter_complaints"
dataset = load_dataset("ought/raft", dataset_name)
dataset["train"][0]
{"Tweet text": "@HMRCcustomers No this is my first job", "ID": 0, "Label": 2}

{'Tweet text': '@HMRCcustomers No this is my first job', 'ID': 0, 'Label': 2}

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Tweet text', 'ID', 'Label'],
        num_rows: 50
    })
    test: Dataset({
        features: ['Tweet text', 'ID', 'Label'],
        num_rows: 3399
    })
})

In [ ]:
def tokenize(batch):
    return tokenizer(batch["Tweet text"], padding=True, truncation=True)

In [ ]:
dataset_encoded = dataset.map(tokenize, batched=True, batch_size=None)

In [ ]:
dataset_encoded

DatasetDict({
    train: Dataset({
        features: ['Tweet text', 'ID', 'Label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50
    })
    test: Dataset({
        features: ['Tweet text', 'ID', 'Label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3399
    })
})

In [ ]:
dataset_encoded.set_format('tf',columns=['input_ids', 'attention_mask', 'token_type_ids', 'Label'])
BATCH_SIZE = 64

def order(inp):
   data = list(inp.values())
   return {
        'input_ids': data[1],
        'attention_mask': data[2],
        'token_type_ids': data[3]
    }, data[0]
train_dataset = tf.data.Dataset.from_tensor_slices(dataset_encoded['train'][:])
train_dataset = train_dataset.batch(BATCH_SIZE).shuffle(1000)
train_dataset = train_dataset.map(order, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices(dataset_encoded['test'][:])
test_dataset = test_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.map(order, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
inp, out = next(iter(train_dataset)) # a batch from train_dataset
print(inp, '\n\n', out)

{'input_ids': <tf.Tensor: shape=(50, 60), dtype=int64, numpy=
array([[  101,  1030, 20287, ...,     0,     0,     0],
       [  101,  1030, 19031, ...,     0,     0,     0],
       [  101,  2065,  1045, ...,     0,     0,     0],
       ...,
       [  101,  1030,  7938, ...,     0,     0,     0],
       [  101, 11990,  2180, ...,     0,     0,     0],
       [  101,  1030,  6568, ...,     0,     0,     0]])>, 'attention_mask': <tf.Tensor: shape=(50, 60), dtype=int64, numpy=
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])>, 'token_type_ids': <tf.Tensor: shape=(50, 60), dtype=int64, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]])>} 

 tf.Tensor(
[2 2 1 1 2 2 2 1 1 2 2 2 2 

In [ ]:
class BERTForClassification(tf.keras.Model):

    def __init__(self, bert_model, num_classes):
        super().__init__()
        self.bert = bert_model
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax')

    def call(self, inputs):
        x = self.bert(inputs)[1]
        return self.fc(x)

In [ ]:
classifier = BERTForClassification(model, num_classes=6)

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

In [ ]:
history = classifier.fit(
    train_dataset,
    epochs=25
)

Epoch 1/25
1/1 [==============================] - 77s 77s/step - loss: 2.0816 - accuracy: 0.0600
Epoch 2/25
1/1 [==============================] - 1s 665ms/step - loss: 1.7909 - accuracy: 0.1200
Epoch 3/25
1/1 [==============================] - 1s 591ms/step - loss: 1.6248 - accuracy: 0.1600
Epoch 4/25
1/1 [==============================] - 1s 593ms/step - loss: 1.3612 - accuracy: 0.3600
Epoch 5/25
1/1 [==============================] - 1s 592ms/step - loss: 1.1676 - accuracy: 0.4800
Epoch 6/25
1/1 [==============================] - 1s 595ms/step - loss: 1.0465 - accuracy: 0.5600
Epoch 7/25
1/1 [==============================] - 1s 593ms/step - loss: 0.9250 - accuracy: 0.6200
Epoch 8/25
1/1 [==============================] - 1s 596ms/step - loss: 0.8970 - accuracy: 0.6800
Epoch 9/25
1/1 [==============================] - 1s 587ms/step - loss: 0.8203 - accuracy: 0.6600
Epoch 10/25
1/1 [==============================] - 1s 609ms/step - loss: 0.7790 - accuracy: 0.6800
Epoch 11/25
1/1 [===

In [ ]:
classifier.evaluate(test_dataset)

NameError: name 'classifier' is not defined